# OCSI on UCF-Crime — full Kaggle run

This notebook clones the OCSI repo and processes **all discovered videos** across `baseline`, `memory`, and `feedback`. Estimate: ~4–6 h on T4 x2 at `--imgsz 320 --batch-size 32`; ~6–10 h at 640px. Use Kaggle **Save & Run All** for the long run.

**Important:** UCF-Crime has no person boxes or IDs. These runs test the *anomaly/cognitive layer* (frame/video AUC) only — they do **not** validate IDF1/IDsw or per-person behaviour feedback. Use MOT17/MOT20 for the tracking claims.

Attach a UCF-Crime dataset to the notebook (or let `kagglehub` download one), and enable a **GPU T4 x2**.

In [ ]:
# Cell 1 — clone or update the repo
!if [ -d /kaggle/working/OCSI/.git ]; then cd /kaggle/working/OCSI && git pull --ff-only; else git clone https://github.com/agdaniel10/OCSI.git /kaggle/working/OCSI; fi

In [ ]:
# Cell 2 — dependencies (torch/torchvision/opencv are preinstalled on Kaggle)
!pip -q install ultralytics pyyaml opencv-python pandas matplotlib
!cd /kaggle/working/OCSI && python -c "import ocsi; print('OCSI import OK')"

In [ ]:
# Cell 3 — locate the dataset
# If the dataset is attached via 'Add Data', this keeps /kaggle/input and lets the runner auto-discover videos.
data_dir = "/kaggle/input"

# Optional: download a public Kaggle copy instead. Comment out if you already attached the data.
try:
    import kagglehub
    data_dir = kagglehub.dataset_download("sabbirahmad653/ucf-crime-datasets")
except Exception as exc:
    print("kagglehub not used; using attached dataset:", exc)

print("UCF data dir:", data_dir)

In [ ]:
# Cell 4 — FULL DATASET run (~4-6 h on T4 x2 with 320px + batch 32)
# Omit --limit-videos / --smoke to process all discovered videos.
# --resume makes the run crash-safe: completed video/stage pairs are skipped.
!cd /kaggle/working/OCSI && python kaggle_ucf_crime.py --data-dir "{data_dir}" --frames 300 --stride 5 --imgsz 320 --batch-size 32 --stages baseline,memory,feedback --resume --out /kaggle/working/ocsi_ucf_outputs --cache /kaggle/working/ocsi_ucf_cache

### Alternatives (do not run both)

- For maximum resolution, replace `--imgsz 320 --batch-size 32` with `--imgsz 640 --batch-size 16` and budget ~6–10 h.
- To continue an interrupted run, simply re-run Cell 4 unchanged; `--resume` skips finished video/stage pairs.
- To process every source frame instead of 300 sampled frames, use `--frames 100000 --stride 1` — that is ~13–14M frames and needs multiple sessions.

In [ ]:
# Cell 6 — inspect outputs
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image

out = Path("/kaggle/working/ocsi_ucf_outputs")
for name in ["ucf_crime_summary.csv", "ucf_crime_frame_scores.csv", "ucf_crime_results.json"]:
    path = out / name
    print(name, "->", path, "exists:", path.exists())

if (out / "ucf_crime_summary.csv").exists():
    display(pd.read_csv(out / "ucf_crime_summary.csv"))
if (out / "ucf_crime_results.json").exists():
    results = json.loads((out / "ucf_crime_results.json").read_text())
    print("Video-level normal-vs-abnormal AUC:", results["video_auc"])
    print("Annotations file:", results["annotations_file"] or "(not found)")
for image in ["ucf_crime_auc.png", "ucf_crime_scores.png"]:
    if (out / image).exists():
        display(Image(filename=str(out / image)))

### One-line alternatives (Kaggle code cell)

```python
!git clone https://github.com/agdaniel10/OCSI.git /kaggle/working/OCSI
!cd /kaggle/working/OCSI && python kaggle_ucf_crime.py --frames 300 --stride 5 --imgsz 320 --batch-size 32 --stages baseline,memory,feedback --resume
```

See `KAGGLE_UCF_CRIME.md` in the repo for all flags and dataset slugs.